# Data-growth trials

Read the fixed split, saved trials, and selected classifier from the original run or bundled results.

These trials share one validation set. Earlier test results used different data.

In [ ]:
import csv
from html import escape
import json
import os

from IPython.display import HTML, display
from PIL import Image
import torch

from train import PROJECT_DIR, REPO_DIR, build_classifier, load_model_checkpoint
from growth_training import ARCHITECTURE, run_iterations

torch.set_num_threads(4)
os.environ["TORCH_HOME"] = str(PROJECT_DIR / ".cache" / "torch")
os.environ["MPLCONFIGDIR"] = str(PROJECT_DIR / ".cache" / "matplotlib")
iteration_dir = PROJECT_DIR / "runs/iterations/data_growth_20261108"
RUN_ITERATIONS = False

if RUN_ITERATIONS:
    run_iterations(iteration_dir)
elif not (iteration_dir / "split_summary.json").exists():
    iteration_dir = REPO_DIR / "results/data_growth_20261108"
print("Iteration directory:", iteration_dir)

In [ ]:
split = json.loads((iteration_dir / "split_summary.json").read_text())
status = json.loads((iteration_dir / "holdout_status.json").read_text())
print("Countries:", split["num_classes"])
print("Audited images:", split["audited_images"])
for role, count in split["split_counts"].items():
    print(role + ":", count)
print("Final holdout status:", status["status"])
print("Split fingerprint:", split["data_fingerprint"])

In [ ]:
summary_path = iteration_dir / "development_summary.csv"
if summary_path.exists():
    with summary_path.open(newline="", encoding="utf-8") as stream:
        summary = list(csv.DictReader(stream))
    headers = ["Stage", "Training images", "Validation images", "Weighted loss", "Mean macro F1", "Mean accuracy", "Mean validation loss"]
    body = []
    for row in summary:
        values = [
            row["stage"], row["training_images"], row["validation_images"], row["weighted_loss"],
            f"{float(row['validation_macro_f1_mean']):.3f}",
            f"{float(row['validation_accuracy_mean']):.2%}",
            f"{float(row['validation_loss_mean']):.4f}",
        ]
        body.append("<tr>" + "".join("<td>" + escape(value) + "</td>" for value in values) + "</tr>")
    display(HTML("<table><tr>" + "".join("<th>" + value + "</th>" for value in headers) + "</tr>" + "".join(body) + "</table>"))
else:
    print("The larger-data fits have not finished yet.")
print("All numbers above are development validation results. No final-test score is shown.")

In [ ]:
figure = iteration_dir / "development_comparison.png"
if figure.exists():
    display(Image.open(figure))
else:
    print("The comparison plot will be available after all predeclared fits finish.")

In [ ]:
selection_path = iteration_dir / "development_selection.json"
if selection_path.exists():
    selection = json.loads(selection_path.read_text())
    print("Selected stage:", selection["stage"])
    print("Selection criterion: mean validation macro F1; ties use lower validation loss.")
    checkpoint_path = iteration_dir / "selected_development/model.pt"
    if checkpoint_path.exists():
        model, checkpoint = load_model_checkpoint(checkpoint_path)
        print(model.fc)
        classes = checkpoint["classes"]
    else:
        print(build_classifier(2048, split["num_classes"], **ARCHITECTURE))
        classes = split["classes"]
        print("The development checkpoint remains in the original workspace.")
    print("Supported countries:", ", ".join(classes))
else:
    print("No continuation configuration has been selected yet.")

## Final fitting

Use all training images and retain the validation set. Final evaluation runs separately after candidate fitting. See `TRAINING_PLAN.md` for recorded settings.

The two seeds share one grouped split. Geographic independence is unverified.